In [6]:
import io
import torch

from pathlib import Path
from pprint import pprint
from fairseq import checkpoint_utils, options, tasks
from contextlib import redirect_stdout, redirect_stderr
from utils import REPO_ROOT, preprocess_mask_predict_data, trace_mask_predict_iterations

In [7]:
prep_result = preprocess_mask_predict_data(
    input_dir=REPO_ROOT / "input",
    output_dir=REPO_ROOT / "output",
    model_dir=REPO_ROOT / "checkpoints" / "maskPredict_en_de",
    source_lang="en",
    target_lang="de",
    run_name="en_de_demo",
    workers=10,
)

In [8]:
trace_result = trace_mask_predict_iterations(
    data_bin_dir=prep_result["data_bin_dir"],
    output_dir=REPO_ROOT / "output",
    model_dir=REPO_ROOT / "checkpoints" / "maskPredict_en_de",
    source_lang="en",
    target_lang="de",
    subset="test",
    run_name="en_de_demo",
    decoding_iterations=10,
    length_beam=5,
    max_sentences=20,
)

for record in trace_result["records"]:
    print(f"[{record['id']}]")
    print("src:", record["source"])
    print("tgt:", record["target"])
    print("hyp:", record["hypothesis"])
    print("iterations:")
    for step in record["iterations"]:
        print(f"iter {step['iteration']:02d} (masked={step['masked_tokens']:>2}): {step['text']}")


[0]
src: The capital of France is Paris .
tgt: Die Hauptstadt Frankreichs ist Paris .
hyp: Die Hauptstadt Frankreichs ist Paris .
iterations:
iter 00 (masked= 0): Die Hauptstadt Frankreichs ist Paris .
iter 01 (masked= 6): Die Hauptstadt Frankreichs ist Paris .
iter 02 (masked= 5): Die Hauptstadt Frankreichs ist Paris .
iter 03 (masked= 4): Die Hauptstadt Frankreichs ist Paris .
iter 04 (masked= 4): Die Hauptstadt Frankreichs ist Paris .
iter 05 (masked= 3): Die Hauptstadt Frankreichs ist Paris .
iter 06 (masked= 2): Die Hauptstadt Frankreichs ist Paris .
iter 07 (masked= 2): Die Hauptstadt Frankreichs ist Paris .
iter 08 (masked= 1): Die Hauptstadt Frankreichs ist Paris .
iter 09 (masked= 0): Die Hauptstadt Frankreichs ist Paris .
[1]
src: The capital of Germany is Berlin .
tgt: Die Hauptstadt Deutschlands ist Berlin .
hyp: Hauptstadt Deutschlands ist Berlin .
iterations:
iter 00 (masked= 0): Hauptstadt Deutschlands ist Berlin .
iter 01 (masked= 5): Hauptstadt Deutschlands ist Berlin 

In [ ]:
def get_encoder_output(sentence: str, data_bin_dir=prep_result["data_bin_dir"], model_dir=REPO_ROOT / "checkpoints" / "maskPredict_en_de", source_lang: str = "en", target_lang: str = "de"):
    data_bin_dir = Path(data_bin_dir)
    checkpoint_path = Path(model_dir) / "checkpoint_best.pt"
    cli_args = [
        str(data_bin_dir),
        "--path", str(checkpoint_path),
        "--task", "translation_self",
        "--source-lang", source_lang,
        "--target-lang", target_lang,
        "--remove-bpe",
        "--max-sentences", "20",
        "--decoding-iterations", "10",
        "--decoding-strategy", "mask_predict",
        "--length-beam", "5",
        "--gen-subset", "test",
        "--cpu",
    ]

    parser = options.get_generation_parser()
    args = options.parse_args_and_arch(parser, input_args=cli_args)

    task = tasks.setup_task(args)
    models, _ = checkpoint_utils.load_model_ensemble(
            args.path.split(":"),
            arg_overrides=eval(args.model_overrides),
            task=task)

    model = models[0].cpu()
    model.eval()

    src_tokens = task.source_dictionary.encode_line(sentence, add_if_not_exist=False).long().unsqueeze(0)
    src_lengths = torch.LongTensor([src_tokens.size(1)])

    with torch.no_grad():
        encoder_out = model.encoder(src_tokens=src_tokens, src_lengths=src_lengths)

    return {
        "sentence": sentence,
        "src_tokens": src_tokens,
        "src_lengths": src_lengths,
        "encoder_out": {
            "encoder_out": encoder_out["encoder_out"].detach().cpu(),
            "encoder_padding_mask": None if encoder_out["encoder_padding_mask"] is None else encoder_out["encoder_padding_mask"].detach().cpu(),
            "predicted_lengths": encoder_out["predicted_lengths"].detach().cpu(),
        },
    }


IndentationError: unexpected indent (4267743447.py, line 22)

In [ ]:
encoder_example = get_encoder_output(
    "China is a large country in Asia , and the capital of the country is Beijing .",
)

print("sentence:", encoder_example["sentence"])
print("src_tokens shape:", tuple(encoder_example["src_tokens"].shape))
print("src_lengths:", encoder_example["src_lengths"].tolist())

for key, value in encoder_example["encoder_out"].items():
    if torch.is_tensor(value):
        print(f"{key}: tensor{tuple(value.shape)}")
    elif isinstance(value, list):
        summary = []
        for item in value:
            if torch.is_tensor(item):
                summary.append(f"tensor{tuple(item.shape)}")
            else:
                summary.append(type(item).__name__)
        print(f"{key}: {summary}")
    else:
        print(f"{key}: {value}")


NameError: name 'warnings' is not defined

In [ ]:
state = torch.load("./checkpoints/maskPredict_en_de/checkpoint_best.pt", map_location="cpu", weights_only=False)

print(state.keys())

args = state["args"]
print(args)


dict_keys(['args', 'model', 'optimizer_history', 'extra_state', 'last_optimizer_state'])
Namespace(adam_betas='(0.9, 0.999)', adam_eps=1e-06, adaptive_softmax_cutoff=None, adaptive_softmax_dropout=0, arch='bert_transformer_seq2seq', attention_dropout=0.0, best_checkpoint_metric='loss', bilm_add_bos=False, bilm_attention_dropout=0.0, bilm_mask_last_state=False, bilm_model_dropout=0.1, bilm_relu_dropout=0.0, bpe=None, bucket_cap_mb=25, clip_norm=25, cpu=False, criterion='label_smoothed_length_cross_entropy', curriculum=0, data=['data-bin/wmt16.s2s.en-de.dist_only'], dataset_impl=None, ddp_backend='c10d', decoder_attention_heads=8, decoder_embed_dim=512, decoder_embed_path=None, decoder_embed_scale=None, decoder_ffn_embed_dim=2048, decoder_input_dim=512, decoder_layers=6, decoder_learned_pos=False, decoder_normalize_before=False, decoder_output_dim=512, device_id=0, disable_validation=False, distributed_backend='nccl', distributed_init_method='tcp://learnfair1412:54186', distributed_no_sp

In [ ]:
for k in [
    "arch",
    "encoder_layers",
    "encoder_embed_dim",
    "encoder_ffn_embed_dim",
    "encoder_attention_heads",
    "decoder_layers",
    "decoder_embed_dim",
    "decoder_ffn_embed_dim",
    "decoder_attention_heads",
    "share_all_embeddings",
    "task",
]:
    if hasattr(args, k):
        print(k, "=", getattr(args, k))

arch = bert_transformer_seq2seq
encoder_layers = 6
encoder_embed_dim = 512
encoder_ffn_embed_dim = 2048
encoder_attention_heads = 8
decoder_layers = 6
decoder_embed_dim = 512
decoder_ffn_embed_dim = 2048
decoder_attention_heads = 8
share_all_embeddings = True
task = translation_self
